In [2]:
cd data/g2p-par


/home/phantom/Desktop/Git/AI_Assignment/assignment-6/class-13and14/data/g2p-par


/home/phantom/Desktop/Git/AI_Assignment/assignment-6/class-13and14/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
!wc *.my *.ph


   2000    5687   59222 dev.my
   2802    8047   83959 test.my
  20000   57336  594183 train.my
   2000    5688   25849 dev.ph
   2802    8048   36532 test.ph
  20000   57346  260356 train.ph
  49604  142152 1060101 total


In [4]:
!head train.my train.ph 


==> train.my <==
ပြာ သာဒ် ဆောင်
ကိုယ် ပိုင် စာ ကြည့် တိုက်
သိန် ဓော ဆား
စား သောက် ဆိုင်
ကြိုး စင်
ဆောင် ကြဉ်း
လော ဘ သက် ကာ ယ
ကမ်း ပါး
ရွှေ ပြည် စိုး
ဟိုက် ဒ ရို ဂျင်

==> train.ph <==
pja' tha' hsaun
kou bain sa kyi. dai'
thein: do: hsa:
sa: thau' hsain
kyou: zin
hsaun kyin:
lo: ba. the' ka ja.
ga- ba:
shwei pji sou:
hai' da- rou gyin


In [5]:
SRC = "my"
TGT = "ph"

!echo "source language: {SRC}"
!echo "target language: {TGT}"


source language: my
target language: ph


In [6]:
!paste train.{SRC} train.{TGT} | head


ပြာ သာဒ် ဆောင်	pja' tha' hsaun
ကိုယ် ပိုင် စာ ကြည့် တိုက်	kou bain sa kyi. dai'
သိန် ဓော ဆား	thein: do: hsa:
စား သောက် ဆိုင်	sa: thau' hsain
ကြိုး စင်	kyou: zin
ဆောင် ကြဉ်း	hsaun kyin:
လော ဘ သက် ကာ ယ	lo: ba. the' ka ja.
ကမ်း ပါး	ga- ba:
ရွှေ ပြည် စိုး	shwei pji sou:
ဟိုက် ဒ ရို ဂျင်	hai' da- rou gyin
paste: write error: Broken pipe
paste: write error


## Moses/PBSMT setup

Moses နဲ့ GIZA++ path တွေကို သတ်မှတ်ပါမယ်။ Direction က Myanmar grapheme (`my`) ကနေ phoneme/romanization (`ph`) ဖြစ်ပါတယ်။


In [7]:
import os
from pathlib import Path

SRC = "my"
TGT = "ph"
PAIR = f"{SRC}-{TGT}"

MOSES = Path("/home/phantom/mosesdecoder")
GIZA = Path("/home/phantom/giza-pp")
WORK = Path.cwd() / "work" / PAIR
TOOLS = WORK / "tools"

print("Direction:", f"{SRC} -> {TGT}")
print("Moses:", MOSES)
print("GIZA++:", GIZA)
print("Work dir:", WORK)


Direction: my -> ph
Moses: /home/phantom/mosesdecoder
GIZA++: /home/phantom/giza-pp
Work dir: /home/phantom/Desktop/Git/AI_Assignment/assignment-6/class-13and14/data/g2p-par/work/my-ph


In [8]:
!mkdir -p {WORK}/lm {WORK}/model {WORK}/mert {WORK}/decode {TOOLS}
!ln -sf /home/phantom/giza-pp/GIZA++-v2/GIZA++ {TOOLS}/GIZA++
!ln -sf /home/phantom/giza-pp/GIZA++-v2/plain2snt.out {TOOLS}/plain2snt.out
!ln -sf /home/phantom/giza-pp/GIZA++-v2/snt2cooc.out {TOOLS}/snt2cooc.out
!ln -sf /home/phantom/giza-pp/mkcls-v2/mkcls {TOOLS}/mkcls
!ls -l {TOOLS}


total 0
lrwxrwxrwx 1 phantom phantom 38 May 22 22:35 GIZA++ -> /home/phantom/giza-pp/GIZA++-v2/GIZA++
lrwxrwxrwx 1 phantom phantom 36 May 22 22:35 mkcls -> /home/phantom/giza-pp/mkcls-v2/mkcls
lrwxrwxrwx 1 phantom phantom 45 May 22 22:35 plain2snt.out -> /home/phantom/giza-pp/GIZA++-v2/plain2snt.out
lrwxrwxrwx 1 phantom phantom 44 May 22 22:35 snt2cooc.out -> /home/phantom/giza-pp/GIZA++-v2/snt2cooc.out


## Build target language model

PBSMT decoder အတွက် target side (`ph`) language model ကို KenLM နဲ့ဆောက်ပါမယ်။


In [9]:
!{MOSES}/bin/lmplz -o 5 --text train.{TGT} --arpa {WORK}/lm/train.{TGT}.arpa
!{MOSES}/bin/build_binary {WORK}/lm/train.{TGT}.arpa {WORK}/lm/train.{TGT}.blm
!ls -lh {WORK}/lm


=== 1/5 Counting and sorting n-grams ===
Reading /home/phantom/Desktop/Git/AI_Assignment/assignment-6/class-13and14/data/g2p-par/train.ph
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 57346 types 1836
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:22032 2:1300661632 3:2438740736 4:3901985024 5:5690395136
Statistics:
1 1836 D1=0.45679 D2=0.896585 D3+=1.23178
2 27905 D1=0.822695 D2=1.27535 D3+=1.11889
3 46127 D1=0.869811 D2=1.29785 D3+=1.58656
4 35930 D1=0.957094 D2=1.6008 D3+=1.9112
5 17169 D1=0.982094 D2=1.79091 D3+=3
Memory estimate for binary LM:
type      kB
probing 2925 assuming -p 1.5
probing 3577 assuming -r models -p 1.5
trie    1334 without quantization
trie     662 assuming -q 8 -b 8 quantization 
trie    1213 assuming -a 22 array pointer compression
trie     541 assuming -a 22 -q 8 -b 8 a

## Train phrase-based SMT model

အောက်က command မှာ assignment direction အတိုင်း `-f my -e ph` သုံးထားပါတယ်။


In [10]:
!time perl {MOSES}/scripts/training/train-model.perl \
  -root-dir {WORK} \
  -corpus train \
  -f {SRC} -e {TGT} \
  -alignment grow-diag-final-and \
  -reordering msd-bidirectional-fe \
  -lm 0:5:{WORK}/lm/train.{TGT}.blm:8 \
  -external-bin-dir {TOOLS} \
  -cores 4 \
  >& {WORK}/training.log

!tail -n 40 {WORK}/training.log
!ls -lh {WORK}/model



real	0m4.221s
user	0m6.539s
sys	0m0.520s
isBSDSplit=0 
Executing: mkdir -p /home/phantom/Desktop/Git/AI_Assignment/assignment-6/class-13and14/data/g2p-par/work/my-ph/model/tmp.18254; ls -l /home/phantom/Desktop/Git/AI_Assignment/assignment-6/class-13and14/data/g2p-par/work/my-ph/model/tmp.18254 
total=20000 line-per-split=5001 
split -d -l 5001 -a 7 /home/phantom/Desktop/Git/AI_Assignment/assignment-6/class-13and14/data/g2p-par/train.ph /home/phantom/Desktop/Git/AI_Assignment/assignment-6/class-13and14/data/g2p-par/work/my-ph/model/tmp.18254/target.split -d -l 5001 -a 7 /home/phantom/Desktop/Git/AI_Assignment/assignment-6/class-13and14/data/g2p-par/train.my /home/phantom/Desktop/Git/AI_Assignment/assignment-6/class-13and14/data/g2p-par/work/my-ph/model/tmp.18254/source.split -d -l 5001 -a 7 /home/phantom/Desktop/Git/AI_Assignment/assignment-6/class-13and14/data/g2p-par/work/my-ph/model/aligned.grow-diag-final-and /home/phantom/Desktop/Git/AI_Assignment/assignment-6/class-13and14/data/

## Tune with development data

Development set (`dev.my`, `dev.ph`) နဲ့ MERT tuning လုပ်ပြီး final tuned `moses.ini` ထုတ်ပါမယ်။


In [11]:
!time perl {MOSES}/scripts/training/mert-moses.pl \
  dev.{SRC} dev.{TGT} \
  {MOSES}/bin/moses \
  {WORK}/model/moses.ini \
  --mertdir {MOSES}/bin \
  --working-dir {WORK}/mert \
  --decoder-flags="-threads 4" \
  >& {WORK}/mert.log

!tail -n 40 {WORK}/mert.log
!ls -lh {WORK}/mert/moses.ini



real	2m47.425s
user	2m59.106s
sys	0m3.206s
Line 1996: Additional reporting took 0.003 seconds total
Line 1996: Translation took 0.030 seconds total
Name:moses	VmPeak:82416 kB	VmRSS:36248 kB	RSSMax:38400 kB	user:7.067	sys:0.529	CPU:7.597	real:1.993
The decoder returns the scores in this order: LexicalReordering0 LexicalReordering0 LexicalReordering0 LexicalReordering0 LexicalReordering0 LexicalReordering0 Distortion0 LM0 WordPenalty0 PhrasePenalty0 TranslationModel0 TranslationModel0 TranslationModel0 TranslationModel0
Executing: gzip -f run3.best100.out
Scoring the nbestlist.
exec: /home/phantom/Desktop/Git/AI_Assignment/assignment-6/class-13and14/data/g2p-par/work/my-ph/mert/extractor.sh
Executing: /home/phantom/Desktop/Git/AI_Assignment/assignment-6/class-13and14/data/g2p-par/work/my-ph/mert/extractor.sh > extract.out 2> extract.err
Executing: \cp -f init.opt run3.init.opt
exec: /home/phantom/mosesdecoder/bin/mert -d 14  --sctype BLEU --scconfig case:true --ffile run1.features.dat,r

## Decode test data

Tuned model နဲ့ `test.my` ကို decode လုပ်ပြီး output ကို `test.decoded.ph` အဖြစ်သိမ်းပါမယ်။


In [12]:
!time {MOSES}/bin/moses \
  -f {WORK}/mert/moses.ini \
  -threads 4 \
  < test.{SRC} \
  > {WORK}/decode/test.decoded.{TGT} \
  2> {WORK}/decode/decode.log

!head -n 10 {WORK}/decode/test.decoded.{TGT}
!wc -l test.{SRC} test.{TGT} {WORK}/decode/test.decoded.{TGT}



real	0m1.641s
user	0m3.951s
sys	0m0.598s
te' te' pjaun 
ka' pi. 
shoun. me. 
njhin: ban: 
mun: man 
nge me 
sha sha hpwei hpwei 
pa: bjin: 
u. ma- kwe: thai' ma- pje' 
a- si' 
  2802 test.my
  2802 test.ph
  2802 /home/phantom/Desktop/Git/AI_Assignment/assignment-6/class-13and14/data/g2p-par/work/my-ph/decode/test.decoded.ph
  8406 total


## BLEU score

Reference `test.ph` နဲ့ decoded output ကို `multi-bleu.perl` နဲ့ score တွက်ပါမယ်။


In [15]:
!perl {MOSES}/scripts/generic/multi-bleu.perl \
  test.{TGT} \
  < {WORK}/decode/test.decoded.{TGT} \
  | tee {WORK}/decode/test.multi-bleu.{PAIR}.txt


It is not advisable to publish scores from multi-bleu.perl.  The scores depend on your tokenizer, which is unlikely to be reproducible from your paper or consistent across research groups.  Instead you should detokenize then use mteval-v14.pl, which has a standard tokenization.  Scores from multi-bleu.perl can still be used for internal purposes when you have a consistent tokenizer.
BLEU = 69.59, 85.2/72.6/64.5/58.8 (BP=1.000, ratio=1.000, hyp_len=8049, ref_len=8048)


In [14]:
!cat {WORK}/decode/test.multi-bleu.{PAIR}.txt


BLEU = 69.59, 85.2/72.6/64.5/58.8 (BP=1.000, ratio=1.000, hyp_len=8049, ref_len=8048)
